In [15]:
import sys
sys.path.append("..")

import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("./data/train_cleaned.csv")
print(f"Loaded {len(df):,} rows")

%load_ext autoreload
%autoreload 2

Loaded 891 rows
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
X = df[['Pclass', 'Sex', 'Age', 'Fare']]
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
# Scaling features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Converting to tensors, y must have the shape [N, 1] and [N]
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

In [18]:
# layer = nn.Linear(in_features=4, out_features=1)
model = nn.Sequential(
    nn.Linear(in_features=4, out_features=1),
    nn.Sigmoid(),
)

prob = model(X_train_tensor)
print(prob)

tensor([[0.4022],
        [0.5738],
        [0.4982],
        [0.5431],
        [0.6090],
        [0.2262],
        [0.4054],
        [0.5048],
        [0.5281],
        [0.5165],
        [0.3908],
        [0.4414],
        [0.5007],
        [0.6101],
        [0.5180],
        [0.3164],
        [0.3363],
        [0.1560],
        [0.4920],
        [0.5199],
        [0.5728],
        [0.2964],
        [0.4689],
        [0.6261],
        [0.2462],
        [0.5882],
        [0.4610],
        [0.5180],
        [0.3235],
        [0.5172],
        [0.3665],
        [0.2536],
        [0.5143],
        [0.3498],
        [0.5284],
        [0.4559],
        [0.6907],
        [0.4879],
        [0.3110],
        [0.4946],
        [0.5292],
        [0.5281],
        [0.5237],
        [0.5817],
        [0.5249],
        [0.5519],
        [0.5212],
        [0.5732],
        [0.5324],
        [0.5020],
        [0.5000],
        [0.4786],
        [0.1960],
        [0.5284],
        [0.4862],
        [0

In [19]:
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

for i in range(1000):

    optimizer.zero_grad()
    output = model(X_train_tensor)
    loss = loss_fn(output, y_train_tensor)
    loss.backward()
    optimizer.step()
    if i % 100 == 0:
        print(loss.item())

0.7580451369285583
0.4602581858634949
0.46025487780570984
0.46025487780570984
0.46025487780570984
0.46025487780570984
0.46025487780570984
0.46025487780570984
0.46025487780570984
0.46025487780570984


In [20]:
with torch.no_grad():
    pred = model(X_test_tensor)
    rounded_pred = (pred > 0.5).float()

    matches = rounded_pred == y_test_tensor

    correct_count = matches.sum().item()
    total_count = len(y_test_tensor)

    acc = correct_count / total_count
    print(f"Accuracy: {acc * 100:.2f}%")


Accuracy: 79.89%
